In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Burden Test Results

In [ ]:
def create_single_volcano_plot(file_path, output_path='volcano_plot.png', top_hits = None):
    """
    Create a single volcano plot for all genetic variants.
    
    Parameters:
    -----------
    file_path : str
        Path to the tab-delimited text file with genetic data
    output_path : str
        Path to save the output plot (default: 'volcano_plot.png')
    """
    # Read the data
    data = pd.read_csv(file_path, sep='\t')

    filtered_data = data[data['BETA'].abs() >= 0]
    
    # Set up the plot
    plt.figure(figsize=(10, 8))
    
    # Create scatter plot
    scatter = plt.scatter(
        filtered_data['BETA'],
        filtered_data['LOG10P'],
        c=pd.factorize(filtered_data['Gene'])[0],  # Color by gene
        cmap='viridis',
        alpha=0.8,
        s=70  # Point size
    )
    
    # Add labels and title
    plt.xlabel('Effect Size (Beta)', fontsize=14)
    plt.ylabel('-log10(P-value)', fontsize=14)
    plt.title('Volcano Plot of Genetic Variants', fontsize=16)
    
    # Add grid
    plt.grid(True, alpha=0.3)

    top_hits_data = top_hits if top_hits is not None else filtered_data.nlargest(10, 'LOG10P')

    for _, row in top_hits_data.iterrows():
        plt.annotate(
            row['Gene'],
            (row['BETA'], row['LOG10P']),
            xytext=(-30, 10),
            textcoords= 'offset points',
            fontsize=8
        )
    
    # Save the plot
    plt.tight_layout()
    #plt.savefig(output_path, dpi=300)
    #print(f"Volcano plot generated and saved to {output_path}")
    
    # Optionally display the plot
    plt.show()

In [ ]:
latent_data = pd.read_csv("./data/burden_test_latent_1025/latent_significant_variants.txt", sep='\t')

# filter to not include low effect sizes (> 50  or < -50)
#data = data[(data['BETA'] > 50) | (data['BETA'] < -50)]

latent_top_hits = latent_data.nlargest(30, 'LOG10P')
latent_top_hits_list = latent_top_hits["Gene"].unique()

create_single_volcano_plot("./data/burden_test_latent_1025/latent_significant_variants.txt", output_path="./data/burden_test_latent_1025/burden_volcano.png", top_hits = latent_top_hits)

In [ ]:
data = pd.read_csv("./data/burden_test_1025/significant_variants.txt", sep='\t')

# drop phenotypes containing "hematocrit"
data = data[~data['Phenotype'].str.contains("hematocrit", case=False, na=False)]

# filter to not include low effect sizes (> 50  or < -50)
#data = data[(data['BETA'] > 50) | (data['BETA'] < -50)]

top_hits = data.nlargest(50, 'LOG10P')
top_hits_list = top_hits["Gene"].unique()

create_single_volcano_plot("./data/burden_test_1025/significant_variants.txt", output_path="./data/burden_test_1025/burden_volcano.png", top_hits = top_hits)

In [ ]:
# compare the two sets of top hits
common_genes = set(latent_data["Gene"]).intersection(set(data["Gene"]))
print("Common genes in both analyses:", common_genes)

# Which phenotypes are associated with these genes?
for gene in common_genes:
    latent_phenotypes = latent_data[latent_data["Gene"] == gene]["Phenotype"].unique()
    standard_phenotypes = data[data["Gene"] == gene]["Phenotype"].unique()
    print(f"Gene: {gene}")
    print(f"  Latent Phenotypes: {latent_phenotypes}", "LOGP:", latent_data[latent_data["Gene"] == gene]["LOG10P"].values, "Betas:", latent_data[latent_data["Gene"] == gene]["BETA"].values)
    print(f"  Standard Phenotypes: {standard_phenotypes}", "LOGP:", data[data["Gene"] == gene]["LOG10P"].values, "Betas:", data[data["Gene"] == gene]["BETA"].values)

In [ ]:
# Venn diagram of significant hits
from matplotlib_venn import venn2

plt.figure(figsize=(8, 6))
venn2(
    subsets = (len(set(latent_data["Gene"])) - len(common_genes), 
               len(set(data["Gene"])) - len(common_genes), 
               len(common_genes)),
    set_labels = ('Latent Analysis', 'Standard Analysis'),
    set_colors=('skyblue', 'lightgreen'),
    alpha=0.7
)
plt.title("Venn Diagram of Significant Genes", fontsize=16)
plt.tight_layout()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming your data is already loaded as latent_data and data
# If not, load like: latent_data = pd.read_csv('latent_results.csv')

# Calculate p-values from LOG10P
latent_data['P'] = 10**(-latent_data['LOG10P'])
data['P'] = 10**(-data['LOG10P'])

# Calculate Bonferroni threshold
# You'll need to tell me the total number of tests
# For now, let's use a placeholder
n_tests_latent = 304000   # or use the actual number
n_tests_t1 = 171000  # or use the actual number
bonf_threshold_latent = 0.05 / (n_tests_latent + n_tests_t1)
bonf_threshold_t1 = bonf_threshold_latent

# Filter for significant hits
latent_sig = latent_data[latent_data['P'] < bonf_threshold_latent].copy()
t1_sig = data[data['P'] < bonf_threshold_t1].copy()

print("="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"Total tests performed:")
print(f"  Latent dimensions: {n_tests_latent:,}")
print(f"  T1 metrics: {n_tests_t1:,}")
print(f"\nBonferroni threshold:")
print(f"  Latent: {bonf_threshold_latent:.2e}")
print(f"  T1: {bonf_threshold_t1:.2e}")
print(f"\nSignificant genes:")
print(f"  Latent dimensions: {latent_sig['Gene'].nunique()}")
print(f"  T1 metrics: {t1_sig['Gene'].nunique()}")
print(f"\nSignificant associations:")
print(f"  Latent dimensions: {len(latent_sig)}")
print(f"  T1 metrics: {len(t1_sig)}")

print("\n" + "="*80)
print("TOP 10 GENES - LATENT DIMENSIONS")
print("="*80)
# Get top gene per phenotype (most significant variant per gene)
latent_top = latent_sig.sort_values('P').groupby('Gene').first().reset_index()
latent_top_10 = latent_top.nsmallest(10, 'P')[['Gene', 'Phenotype', 'LOG10P', 'BETA', 'P']]
print(latent_top_10.to_string(index=False))

print("\n" + "="*80)
print("TOP 10 GENES - T1 METRICS")
print("="*80)
t1_top = t1_sig.sort_values('P').groupby('Gene').first().reset_index()
t1_top_10 = t1_top.nsmallest(10, 'P')[['Gene', 'Phenotype', 'LOG10P', 'BETA', 'P']]
print(t1_top_10.to_string(index=False))

print("\n" + "="*80)
print("OVERLAP ANALYSIS")
print("="*80)
latent_genes = set(latent_sig['Gene'].unique())
t1_genes = set(t1_sig['Gene'].unique())
overlap_genes = latent_genes.intersection(t1_genes)
print(f"Genes significant in BOTH analyses: {len(overlap_genes)}")
if len(overlap_genes) > 0:
    print(f"Overlapping genes: {', '.join(sorted(overlap_genes)[:20])}")
    if len(overlap_genes) > 20:
        print(f"  ... and {len(overlap_genes)-20} more")

print("\n" + "="*80)
print("GWAS GENE CHECK")
print("="*80)
gwas_genes = ['CAMK2D', 'SOD2', 'GSS', 'CALU', 'ARHGAP27', 'C2ORF88']
for gene in gwas_genes:
    in_t1 = gene in t1_genes
    in_latent = gene in latent_genes
    print(f"{gene:12s}: T1={'YES' if in_t1 else 'NO':3s}  Latent={'YES' if in_latent else 'NO':3s}")

print("\n" + "="*80)
print("EFFECT SIZE COMPARISON")
print("="*80)
print("Rare variants (burden tests):")
print(f"  T1 metrics: β range [{t1_sig['BETA'].min():.3f}, {t1_sig['BETA'].max():.3f}], median={t1_sig['BETA'].median():.3f}")
print(f"  Latent dims: β range [{latent_sig['BETA'].min():.3f}, {latent_sig['BETA'].max():.3f}], median={latent_sig['BETA'].median():.3f}")
print("\nCommon variants (GWAS - from your data):")
print(f"  T1 metrics: β range [1.7, 2.3] ms per allele (from GWAS)")

print("\n" + "="*80)
print("LATENT DIMENSION BREAKDOWN")
print("="*80)
latent_by_pheno = latent_sig.groupby('Phenotype').agg({
    'Gene': 'nunique',
    'Variant_ID': 'count',
    'LOG10P': 'max'
}).sort_values('Gene', ascending=False)
latent_by_pheno.columns = ['Unique_Genes', 'N_Associations', 'Max_LOG10P']
print(latent_by_pheno)

In [ ]:
# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Number of significant genes by phenotype
ax1 = axes[0, 0]
t1_counts = t1_sig.groupby('Phenotype')['Gene'].nunique().sort_values(ascending=False)
latent_counts = latent_sig.groupby('Phenotype')['Gene'].nunique().sort_values(ascending=False)
x = np.arange(max(len(t1_counts), len(latent_counts)))
ax1.bar(x[:len(t1_counts)] - 0.2, t1_counts.values, 0.4, label='T1 Metrics', alpha=0.8)
ax1.bar(x[:len(latent_counts)] + 0.2, latent_counts.values, 0.4, label='Latent Dims', alpha=0.8)
ax1.set_xlabel('Phenotype Rank')
ax1.set_ylabel('Number of Significant Genes')
ax1.set_title('Significant Genes by Phenotype')
ax1.legend()

# 2. Effect size distribution
ax2 = axes[0, 1]
ax2.hist(t1_sig['BETA'], bins=50, alpha=0.5, label='T1 Metrics', density=True)
ax2.hist(latent_sig['BETA'], bins=50, alpha=0.5, label='Latent Dims', density=True)
ax2.set_xlabel('Effect Size (β)')
ax2.set_ylabel('Density')
ax2.set_title('Effect Size Distribution')
ax2.legend()
ax2.axvline(0, color='black', linestyle='--', alpha=0.3)

# 3. Allele frequency distribution
ax3 = axes[1, 0]
ax3.hist(t1_sig['Allele_Freq'], bins=50, alpha=0.5, label='T1 Metrics', density=True)
ax3.hist(latent_sig['Allele_Freq'], bins=50, alpha=0.5, label='Latent Dims', density=True)
ax3.set_xlabel('Minor Allele Frequency')
ax3.set_ylabel('Density')
ax3.set_title('MAF Distribution of Significant Variants')
ax3.legend()
ax3.set_xlim(0, 0.01)  # Focus on rare variants

# 4. -log10(P) vs effect size
ax4 = axes[1, 1]
ax4.scatter(t1_sig['BETA'], t1_sig['LOG10P'], alpha=0.3, s=10, label='T1 Metrics')
ax4.scatter(latent_sig['BETA'], latent_sig['LOG10P'], alpha=0.3, s=10, label='Latent Dims')
ax4.set_xlabel('Effect Size (β)')
ax4.set_ylabel('-log10(P)')
ax4.set_title('Significance vs Effect Size')
ax4.legend()
ax4.axvline(0, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('rare_variant_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Export tables for manuscript
print("\n" + "="*80)
print("EXPORTING TABLES")
print("="*80)

# Table 1: Top genes for supplement
top_genes_combined = pd.concat([
    latent_top_10.assign(Analysis='Latent'),
    t1_top_10.assign(Analysis='T1')
])
top_genes_combined.to_csv('top_rare_variant_genes.csv', index=False)
print("Saved: top_rare_variant_genes.csv")

# Table 2: All significant genes
latent_sig.to_csv('latent_significant_rare_variants.csv', index=False)
t1_sig.to_csv('t1_significant_rare_variants.csv', index=False)
print("Saved: latent_significant_rare_variants.csv")
print("Saved: t1_significant_rare_variants.csv")

# Table 3: Overlap genes
if len(overlap_genes) > 0:
    overlap_df = pd.DataFrame({
        'Gene': sorted(overlap_genes),
        'In_T1': True,
        'In_Latent': True
    })
    overlap_df.to_csv('overlapping_genes.csv', index=False)
    print("Saved: overlapping_genes.csv")

# Heretability Results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
from matplotlib import rcParams

rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Helvetica', 'Verdana']

# Set global font size parameters
SMALL_FONT = 20    # Was 9-10
MEDIUM_FONT = 20   # Was 12
LARGE_FONT = 24    # Was 14

# Increase all font sizes globally
plt.rc('font', size=MEDIUM_FONT)          # Default text sizes
plt.rc('axes', titlesize=LARGE_FONT)      # Axes title
plt.rc('axes', labelsize=MEDIUM_FONT)     # X and Y labels
plt.rc('xtick', labelsize=SMALL_FONT)     # X tick labels
plt.rc('ytick', labelsize=SMALL_FONT)     # Y tick labels
plt.rc('legend', fontsize=SMALL_FONT)     # Legend
plt.rc('figure', titlesize=LARGE_FONT)    # Figure title

# Define the data
latent_data = [
    {"Phenotype": "Latent_0", "h2": 0.0649, "SE": 0.0273, "p_value": 1.74e-02, "type": "Latent"},
    {"Phenotype": "Latent_1", "h2": 0.0380, "SE": 0.0277, "p_value": 1.70e-01, "type": "Latent"},
    {"Phenotype": "Latent_2", "h2": -0.0088, "SE": 0.0290, "p_value": 7.62e-01, "type": "Latent"},
    {"Phenotype": "Latent_3", "h2": 0.0349, "SE": 0.0259, "p_value": 1.78e-01, "type": "Latent"},
    {"Phenotype": "Latent_4", "h2": 0.0111, "SE": 0.0288, "p_value": 7.00e-01, "type": "Latent"},
    {"Phenotype": "Latent_5", "h2": 0.0186, "SE": 0.0276, "p_value": 5.00e-01, "type": "Latent"},
    {"Phenotype": "Latent_6", "h2": 0.0664, "SE": 0.0304, "p_value": 2.89e-02, "type": "Latent"},
    {"Phenotype": "Latent_7", "h2": 0.0682, "SE": 0.0278, "p_value": 1.42e-02, "type": "Latent"},
    {"Phenotype": "Latent_8", "h2": 0.0369, "SE": 0.0280, "p_value": 1.88e-01, "type": "Latent"},
    {"Phenotype": "Latent_9", "h2": 0.0257, "SE": 0.0250, "p_value": 3.04e-01, "type": "Latent"},
    {"Phenotype": "Latent_10", "h2": 0.0106, "SE": 0.0285, "p_value": 7.10e-01, "type": "Latent"},
    {"Phenotype": "Latent_11", "h2": 0.0627, "SE": 0.0281, "p_value": 2.57e-02, "type": "Latent"},
    {"Phenotype": "Latent_12", "h2": -0.0203, "SE": 0.0255, "p_value": 4.26e-01, "type": "Latent"},
    {"Phenotype": "Latent_13", "h2": 0.0276, "SE": 0.0272, "p_value": 3.10e-01, "type": "Latent"},
    {"Phenotype": "Latent_14", "h2": 0.0120, "SE": 0.0265, "p_value": 6.51e-01, "type": "Latent"},
    {"Phenotype": "Latent_15", "h2": 0.0231, "SE": 0.0296, "p_value": 4.35e-01, "type": "Latent"}
]

t1_data = [
    {"Phenotype": "Mean_T1", "h2": 0.1606, "SE": 0.0365, "p_value": 1.08e-05, "type": "T1"},
    {"Phenotype": "T1_0.25th_Percentile", "h2": 0.0419, "SE": 0.0350, "p_value": 2.31e-01, "type": "T1"},
    {"Phenotype": "T1_1th_Percentile", "h2": 0.0635, "SE": 0.0360, "p_value": 7.78e-02, "type": "T1"},
    {"Phenotype": "T1_25th_Percentile", "h2": 0.1461, "SE": 0.0364, "p_value": 5.98e-05, "type": "T1"},
    {"Phenotype": "T1_50th_Percentile", "h2": 0.1627, "SE": 0.0366, "p_value": 8.78e-06, "type": "T1"},
    {"Phenotype": "T1_75th_Percentile", "h2": 0.1725, "SE": 0.0382, "p_value": 6.32e-06, "type": "T1"},
    {"Phenotype": "T1_99.75th_Percentile", "h2": 0.0781, "SE": 0.0344, "p_value": 2.32e-02, "type": "T1"},
    {"Phenotype": "T1_99th_Percentile", "h2": 0.0715, "SE": 0.0360, "p_value": 4.70e-02, "type": "T1"},
    {"Phenotype": "T1_Standard_Deviation", "h2": 0.0979, "SE": 0.0376, "p_value": 9.22e-03, "type": "T1"}
]

# Combine the data
combined_data = latent_data + t1_data

# Convert to DataFrame
df = pd.DataFrame(combined_data)

# Calculate 95% confidence intervals
df['CI_lower'] = df['h2'] - 1.96 * df['SE']
df['CI_upper'] = df['h2'] + 1.96 * df['SE']

# Map p-values to numeric values for marker size
def categorize_pvalue(p):
    if p < 0.0001:
        return 30, 'o'
    elif p < 0.001:
        return 25, 'o'
    elif p < 0.01:
        return 20, 'o'
    elif p < 0.05:
        return 15, 'o'
    elif p < 0.10:
        return 10, 's'
    else:
        return 5, 's'

# Apply the function to create the numeric and marker columns
df[['p_value_numeric', 'p_value_marker']] = df['p_value'].apply(
    lambda x: pd.Series(categorize_pvalue(x))
)

# Sort the dataframe by h2 value
df = df.sort_values('h2', ascending=False)

# Add a row index for plotting
df = df.reset_index(drop=True)

# Create color mapping for phenotype types
type_colors = {
    "Latent": "#4575B4",  # Blue
    "T1": "#D73027"       # Red
}

# Set up the figure
plt.figure(figsize=(12, 14))

# Create the horizontal forest plot
for i, row in df.iterrows():
    # Plot confidence interval line
    plt.plot([row['CI_lower'], row['CI_upper']], [i, i], 
             color='black', alpha=0.6, zorder=1)
    
    # Plot the point estimate (h2)
    significance_size = 50 + (row['p_value_numeric'] * 20)  # Adjust marker size based on significance
    plt.scatter(row['h2'], i, 
                color=type_colors[row['type']], 
                s=significance_size, 
                zorder=2, 
                edgecolor='black', 
                marker=row['p_value_marker'],
                linewidth=1,
                alpha=0.8)

# Add vertical line at x=0 (null effect)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.7, zorder=0)

# Customize the plot
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.xlabel('Heritability (h²)', fontsize=12, fontweight='bold')
plt.ylabel('Phenotype', fontsize=12, fontweight='bold')
plt.title("Forest Plot of Heritability (h²) Estimates with 95% CI", fontsize=14, fontweight='bold')

# Set y-ticks to show phenotype names
plt.yticks(range(len(df)), df['Phenotype'], fontsize=10)

# # Annotate with h2 values and p-values
# for i, row in df.iterrows():
#     # Add h2 value to the right
#     plt.text(row['CI_upper'] + 0.02, i, 
#              f"{row['h2']:.4f} ({row['P_value']})", 
#              va='center', fontsize=9)
    
#     # Add type label to the left
#     plt.text(-0.3, i, row['type'], va='center', ha='right', 
#              fontsize=9, fontweight='bold', color=type_colors[row['type']])

# Add a legend for phenotype types
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=type_colors['Latent'], edgecolor='black', label='Latent'),
    Patch(facecolor=type_colors['T1'], edgecolor='black', label='T1')
]

# p_value_order = {
    'p < 0.0001': 30,
    'p < 0.001': 25,
    'p < 0.01': 20,
    'p < 0.05': 15,
    'p < 0.10': 10,
    'n.s.': 5,
}
p_value_marker = {
    'p < 0.0001': 'o',
    'p < 0.001': 'o',
    'p < 0.01': 'o',
    'p < 0.05': 'o',
    'p < 0.10': 's',
    'n.s.': 's',
}

# Add a legend for p-value significance
legend_p_vals = []
sizes = []
for p_val, size_factor in sorted(p_value_order.items(), key=lambda x: x[1], reverse=True):
    p_val_display = p_val.replace('<', '≤') if '<' in p_val else p_val
    marker_shape = p_value_marker[p_val]
    
    # Use square for ns, circle for significant
    marker_obj = plt.scatter([], [], 
                           s= (size_factor * 100), 
                           color='gray', 
                           marker=marker_shape,
                           edgecolor='black', 
                           label=p_val_display)
    legend_p_vals.append(marker_obj)

# Create a combined legend
plt.legend(
    handles=legend_elements + legend_p_vals, 
    labels=['Latent', 'T1'] + list(p_value_order.keys()),
    title="Phenotype Type and Significance",
    loc='lower left',
    bbox_to_anchor=(1.0, 0.0)
)

# Set x-axis limits
x_min = min(df['CI_lower'].min() - 0.05, -0.3)
x_max = max(df['CI_upper'].max() + 0.2, 0.3)
plt.xlim(x_min, x_max)

# Tight layout to maximize use of space
plt.tight_layout()

# Save the plot
#plt.savefig('heritability_forest_plot.png', dpi=300, bbox_inches='tight')
plt.show()

# MR

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
import re

# Read the data from file
def create_mr_forest_plot(df, outfile="mr_forest_plot.png"):
    # Convert columns to appropriate data types
    df['ivw_estimate'] = df['ivw_estimate'].astype(float)
    df['ivw_se'] = df['ivw_se'].astype(float)
    df['ivw_pval'] = df['ivw_pval'].astype(float)
    df['n_snps'] = df['n_snps'].astype(int)
    
    # Calculate confidence intervals
    df['ci_lower'] = df['ivw_estimate'] - 1.96 * df['ivw_se']
    df['ci_upper'] = df['ivw_estimate'] + 1.96 * df['ivw_se']
    
    # Clean up exposure and outcome names
    df['gene'] = df['exposure_file'].str.replace('_cis_subset.tsv', '')
    
    # Clean outcome names
    df['outcome'] = df['outcome_file'].str.replace('_gwas.txt', '')
    df['outcome'] = df['outcome'].str.replace('imputed_T1_10itr_erroded_gwas_output.', '')
    
    # Create display label
    df['label'] = df['gene'] + ' → ' + df['outcome']
    
    # Categorize p-values for color coding
    def get_pval_category(p):
        if p < 1e-10:
            return "p < 1e-10"
        elif p < 1e-8:
            return "p < 1e-8"
        elif p < 1e-6:
            return "p < 1e-6"
        else:
            return "p ≥ 1e-6"
    
    df['pval_category'] = df['ivw_pval'].apply(get_pval_category)
    
    # Define marker mapping based on p-values
    pval_marker = {
        "p < 1e-10": 'o',
        "p < 1e-8": 'o',
        "p < 1e-6": 'o',
        "p ≥ 1e-6": 's'  # Square for less significant
    }
    
    df['marker'] = df['pval_category'].map(pval_marker)
    
    # Create a color mapping for p-value categories
    pval_colors = {
        "p < 1e-10": "#0047AB",  # Strong blue
        "p < 1e-8": "#4169E1",   # Royal blue
        "p < 1e-6": "#87CEEB",   # Sky blue
        "p ≥ 1e-6": "#D3D3D3"    # Light gray
    }
    
    # Sort by p-value and select top 30 most significant results
    df_plot = df.sort_values('ivw_pval').head(30).reset_index(drop=True)
    df_plot = df.sort_values('exposure_file').reset_index(drop=True)
    
    # Create the forest plot
    plt.figure(figsize=(12, 14))
    
    # Plot settings
    spacing = 1
    positions = np.arange(len(df_plot)) * spacing
    
    # Create the plot
    for i, row in df_plot.iterrows():
        # Plot confidence interval line
        plt.plot([row['ci_lower'], row['ci_upper']], [positions[i], positions[i]], 
                 color='black', alpha=0.6, zorder=1, linewidth=1.5)
        
        # Determine marker size based on number of SNPs
        marker_size = 60 + min(row['n_snps'] * 1.5, 500)  # Cap the size
        marker_size = 600
        
        # Plot the point estimate
        plt.scatter(row['ivw_estimate'], positions[i], 
                    color=pval_colors[row['pval_category']], 
                    s=marker_size,
                    marker=row['marker'],
                    zorder=2, 
                    edgecolor='black', 
                    linewidth=3,
                    alpha=0.8)

    
    # Add vertical line at x=0 (null effect)
    plt.axvline(x=0, color='gray', linestyle='--', alpha=0.7, zorder=0, linewidth=2)
    
    # Customize the plot
    plt.grid(axis='x', linestyle='--', alpha=0.3, linewidth=1.0)
    plt.xlabel('Effect Size (IVW Estimate)', fontsize=14, fontweight='bold')
    plt.title('Mendelian Randomization Results', fontsize=16, fontweight='bold')
    
    # Set y-ticks to show gene-outcome pairs
    plt.yticks(positions, df_plot['label'], fontsize=10)
    
    # Add effect sizes and p-values as annotations
    # for i, row in df_plot.iterrows():
    #     # Format p-value for display
    #     pval_text = f"p={row['ivw_pval']:.2e}"
        
    #     # Add estimate and p-value
    #     plt.text(row['ci_upper'] + 0.5, positions[i], 
    #              f"{row['ivw_estimate']:.2f} [{row['ci_lower']:.2f}, {row['ci_upper']:.2f}]", 
    #              va='center', fontsize=9)
        
        # Add number of SNPs
        # plt.text(row['ci_lower'] - 2, positions[i], 
        #          f"n={row['n_snps ']}", 
        #          va='center', ha='right', fontsize=9)
    
    # Create legend for p-value categories
    legend_elements = []
    for pval, color in pval_colors.items():
        marker_shape = pval_marker[pval]
        legend_elements.append(plt.scatter([], [], 
                               s=100, 
                               color=color, 
                               marker=marker_shape,
                               edgecolor='black', 
                               linewidth=1.5,
                               label=pval))
    
    # Add legend
    plt.legend(
        handles=legend_elements, 
        labels=list(pval_colors.keys()),
        title="P-value",
        loc='lower right',
        fontsize=10,
        title_fontsize=12
    )
    
    # Set x-axis limits with buffer for annotations
    x_min = min(df_plot['ci_lower'].min() - 5, -5)
    x_max = max(df_plot['ci_upper'].max() + 15, 15)
    plt.xlim(x_min, x_max)
    
    # Set y-axis limits
    plt.ylim(-1, len(df_plot))
    # Set x-axis limits capped with small buffer
    #plt.xlim(-0.75, 0.75)

    # Set y-axis limits
    plt.ylim(-1, len(df_plot))
    # Tight layout with padding
    plt.tight_layout(pad=2.0)
    
    # Save high-resolution plot
    #plt.savefig(outfile, dpi=300, bbox_inches='tight')
    #print(f"Forest plot saved as {outfile}")
    
    # Create a second plot grouped by exposure gene
    #create_grouped_forest_plot(df, outfile="mr_forest_plot_by_gene.png")
    plt.savefig(outfile, format='svg', dpi=300, bbox_inches='tight')
    print(f"Forest plot saved as {outfile}")
    
    return df


In [ ]:
#df = pd.read_csv("./data/MR_significant_hits.txt", sep='\t')
#df = pd.read_csv("./data/significant_associations.txt", sep='\t')

MR_PATH = "./data/MR_T1_preprint_sig.txt"
MR_latent_PATH = "./data/MR_latent_preprint_sig.txt"

latent_data = pd.read_csv(MR_latent_PATH, sep='\t')
t1_data = pd.read_csv(MR_PATH, sep='\t')

#drop rows with "hematocrit" in outcome_file
t1_data = t1_data[~t1_data['outcome_file'].str.contains("hematocrit", case=False, na=False)]

genes_to_include = [
    "FOLH1", "C2", "LRRC37A2", "FUT8", "ENPP2", "DCBLD2", "APOH", "DNAJC9"
]
# df = df[df['exposure_file'].str.contains('|'.join(genes_to_include))]

# #remove duplicates
# df['outcome_file'] = df['outcome_file'].str.replace('imputed_T1_10itr_erroded_gwas_output.', '')
# df = df.drop_duplicates(subset=['exposure_file', 'outcome_file'])

t1_data = t1_data.drop_duplicates(subset=['exposure_file', 'outcome_file'])
latent_data = latent_data.drop_duplicates(subset=['exposure_file', 'outcome_file'])

In [ ]:
def extract_phenotype(filename):
    # Remove the "GWAS_output." prefix
    if filename.startswith('GWAS_output.'):
        phenotype = filename.replace('GWAS_output.', '')
    else:
        phenotype = filename
    
    # Remove "_gwas.txt" suffix if present
    phenotype = re.sub(r'_gwas\.txt$', '', phenotype)
    
    # Remove "T1_" prefix if present
    phenotype = re.sub(r'^T1_', '', phenotype)
    
    # Replace underscores with spaces
    phenotype = phenotype.replace('_', ' ')
    
    # Handle truncated cases (ending with ...)
    phenotype = re.sub(r'\.\.\..*$', '', phenotype)
    
    # Clean up extra spaces
    phenotype = ' '.join(phenotype.split())
    
    return phenotype

# Apply the function to your dataframe
# df['outcome_file'] = df['outcome_file'].apply(extract_phenotype)

t1_data['outcome_file'] = t1_data['outcome_file'].apply(extract_phenotype)
latent_data['outcome_file'] = latent_data['outcome_file'].apply(extract_phenotype)

# df_regressed = df[df['outcome_file'].str.contains('regressed hematocrit hypertension', case=False)]

# Then clean up the outcome_file names by removing the regressed hematocrit phrases
def clean_regressed_names(name):
    # Remove "regressed hematocrit hypertension status" first (longer phrase)
    name = name.replace(' regressed hematocrit hypertension status', '')
    # Then remove "regressed hematocrit" 
    name = name.replace(' regressed hematocrit', '')
    # Clean up any extra spaces
    return ' '.join(name.split())

# df_regressed['outcome_file'] = df_regressed['outcome_file'].apply(clean_regressed_names)
# df_regressed.sort_values('outcome_file')

In [ ]:
latent_data["exposure_file"] = latent_data["exposure_file"].str.replace('_cis_subset.tsv', '')
t1_data["exposure_file"] = t1_data["exposure_file"].str.replace('_cis_subset.tsv', '')

In [ ]:
#overlap genes
common_genes = set(latent_data["exposure_file"]).intersection(set(t1_data["exposure_file"]))
print("Common genes in both analyses:", common_genes)
print("Number of common genes:", len(common_genes))

In [ ]:
for gene in common_genes:
    latent_phenotypes = latent_data[latent_data["exposure_file"] == gene]["outcome_file"].unique()
    standard_phenotypes = t1_data[t1_data["exposure_file"] == gene]["outcome_file"].unique()
    print(f"Gene: {gene}")
    print(f"  Latent Phenotypes: {latent_phenotypes}", "LOGP:", latent_data[latent_data["exposure_file"] == gene]["ivw_pval"].values, "Betas:", latent_data[latent_data["exposure_file"] == gene]["ivw_estimate"].values)

    t1_phenotypes = t1_data[t1_data["exposure_file"] == gene]["outcome_file"].unique()
    print(f"  T1 Phenotypes: {standard_phenotypes}", "LOGP:", t1_data[t1_data["exposure_file"] == gene]["ivw_pval"].values, "Betas:", t1_data[t1_data["exposure_file"] == gene]["ivw_estimate"].values)
    

In [ ]:
# Optional: Print summary statistics
print("T1 MR Summary Statistics:")
print(f"Total number of results: {len(t1_data)}")
print(f"Number of unique exposure genes: {t1_data['exposure_file'].nunique()}")
print(f"Number of unique outcomes: {t1_data['outcome_file'].nunique()}")
print("\nTop 10 most significant results:")
print(t1_data.sort_values('ivw_pval').head(10)[['exposure_file', 'outcome_file', 'ivw_estimate', 'ivw_pval', 'n_snps']])
    
# Optional: Print summary statistics
print("Latent MR Summary Statistics:")
print(f"Total number of results: {len(latent_data)}")
print(f"Number of unique exposure genes: {latent_data['exposure_file'].nunique()}")
print(f"Number of unique outcomes: {latent_data['outcome_file'].nunique()}")
print("\nTop 10 most significant results:")
print(latent_data.sort_values('ivw_pval').head(10)[['exposure_file', 'outcome_file', 'ivw_estimate', 'ivw_pval', 'n_snps']])

In [ ]:
data = latent_data[
    (latent_data["exposure_file"] == "RNF5") & (latent_data["outcome_file"] == "Latent 14") |
    (latent_data["exposure_file"] == "TNFAIP8L2") & (latent_data["outcome_file"] == "Latent 6") |
    (latent_data["exposure_file"] == "DAG1") & (latent_data["outcome_file"] == "Latent 13") |
    (latent_data["exposure_file"] == "RNF5") & (latent_data["outcome_file"] == "Latent 10") |
    (latent_data["exposure_file"] == "FKBPL") & (latent_data["outcome_file"] == "Latent 14") |
    (latent_data["exposure_file"] == "WNT9A") & (latent_data["outcome_file"] == "Latent 3") |
    (latent_data["exposure_file"] == "DPY30") & (latent_data["outcome_file"] == "Latent 12") |
    (latent_data["exposure_file"] == "FKBPL") & (latent_data["outcome_file"] == "Latent 10")
]

create_mr_forest_plot(data, outfile="./data/MR_latent_grant_sig.svg")

#save as svg



In [ ]:
t1_mr_subdata = t1_data[
    (t1_data["exposure_file"] == "RNF5") & (t1_data["outcome_file"] == "1th Percentile") |
    (t1_data["exposure_file"] == "PLEKHO1") & (t1_data["outcome_file"] == "99th Percentile") |
    (t1_data["exposure_file"] == "TNFAIP8L2") & (t1_data["outcome_file"] == "99th Percentile") |
    (t1_data["exposure_file"] == "FOLH1") & (t1_data["outcome_file"] == "99.75th Percentile") |
    (t1_data["exposure_file"] == "FOLH1") & (t1_data["outcome_file"] == "99th Percentile") |
    (t1_data["exposure_file"] == "RNF5") & (t1_data["outcome_file"] == "Standard Deviation") |
    (t1_data["exposure_file"] == "SCGN") & (t1_data["outcome_file"] == "75th Percentile") |
    (t1_data["exposure_file"] == "SERPINC1") & (t1_data["outcome_file"] == "75th Percentile") |
    (t1_data["exposure_file"] == "SCGN") & (t1_data["outcome_file"] == "99th Percentile") |
    t1_data["exposure_file"].isin(["FKBPL", "CTSS", "ECM1", "APOH", "LRRC37A2"])
]

create_mr_forest_plot(t1_mr_subdata, outfile="./data/MR_t1_grant_sig.svg")

In [ ]:
def create_two_dataframe_volcano_plot(df1, df2,
                                     df1_label='Dataset 1',
                                     df2_label='Dataset 2',
                                     overlap_label='Both',
                                     match_col='exposure_file',
                                     beta_col='ivw_estimate',
                                     pval_col='ivw_pval',
                                     key_labels=None,
                                     bonferroni_alpha=0.05,
                                     figsize=(12, 8),
                                     ylim=None,
                                     xlim=None,
                                     title='MR Results Comparison',
                                     save_path=None,
                                     show_overlap=True):
    """
    Create a volcano plot comparing two dataframes with color coding
    
    Parameters:
    -----------
    df1 : pd.DataFrame
        First dataset (e.g., Manual MR)
    df2 : pd.DataFrame
        Second dataset (e.g., KOSMOS MR)
    df1_label : str
        Label for df1 points
    df2_label : str
        Label for df2 points
    overlap_label : str
        Label for overlapping points
    match_col : str
        Column to use for matching between dataframes (e.g., 'exposure_file')
    beta_col : str
        Column name for effect size
    pval_col : str
        Column name for p-values
    key_labels : list, optional
        Specific points to label
    show_overlap : bool
        Whether to show overlapping points as a separate category
    """
    
    # Copy dataframes
    df1 = df1.copy()
    df2 = df2.copy()
    
    # Add source column
    df1['source'] = df1_label
    df2['source'] = df2_label
    
    # Calculate -log10(p-value) for both
    df1['log_p'] = -np.log10(df1[pval_col])
    df2['log_p'] = -np.log10(df2[pval_col])
    
    # Handle infinite values
    df1['log_p'] = df1['log_p'].replace([np.inf, -np.inf], np.nan)
    df2['log_p'] = df2['log_p'].replace([np.inf, -np.inf], np.nan)
    df1 = df1.dropna(subset=['log_p', beta_col])
    df2 = df2.dropna(subset=['log_p', beta_col])
    
    # Identify overlapping and unique entries
    df1_items = set(df1[match_col])
    df2_items = set(df2[match_col])
    
    overlap_items = df1_items & df2_items
    df1_only = df1_items - df2_items
    df2_only = df2_items - df1_items
    
    print(f"{df1_label} only: {len(df1_only)}")
    print(f"{df2_label} only: {len(df2_only)}")
    print(f"Overlapping: {len(overlap_items)}")
    
    # Calculate Bonferroni threshold
    total_unique = len(df1_items | df2_items)
    bonferroni_threshold = -np.log10(bonferroni_alpha / total_unique)
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    if show_overlap:
        # Separate into three categories
        df1_overlap = df1[df1[match_col].isin(overlap_items)]
        df1_unique = df1[df1[match_col].isin(df1_only)]
        df2_overlap = df2[df2[match_col].isin(overlap_items)]
        df2_unique = df2[df2[match_col].isin(df2_only)]
        
        # Plot unique points first (background)
        if len(df1_unique) > 0:
            ax.scatter(df1_unique[beta_col], df1_unique['log_p'], 
                      c='#457B9D', s=80, alpha=0.7, 
                      label=f'{df1_label} only (n={len(df1_only)})', 
                      edgecolors='black', linewidth=0.5, zorder=2)
        
        if len(df2_unique) > 0:
            ax.scatter(df2_unique[beta_col], df2_unique['log_p'], 
                      c='#2A9D8F', s=80, alpha=0.7, 
                      label=f'{df2_label} only (n={len(df2_only)})', 
                      edgecolors='black', linewidth=0.5, zorder=2)
        
        # Plot overlapping points on top (use df1 values for overlap)
        if len(df1_overlap) > 0:
            ax.scatter(df1_overlap[beta_col], df1_overlap['log_p'], 
                      c='#E63946', s=80, alpha=0.7, 
                      label=f'{overlap_label} (n={len(overlap_items)})', 
                      edgecolors='black', linewidth=0.5, zorder=3)
    
    else:
        # Just show two colors - one per dataframe
        ax.scatter(df1[beta_col], df1['log_p'], 
                  c='#E63946', s=80, alpha=0.7, 
                  label=f'{df1_label} (n={len(df1)})', 
                  edgecolors='black', linewidth=0.5, zorder=2)
        
        ax.scatter(df2[beta_col], df2['log_p'], 
                  c='#2A9D8F', s=80, alpha=0.7, 
                  label=f'{df2_label} (n={len(df2)})', 
                  edgecolors='black', linewidth=0.5, zorder=3)
    
    # Add threshold lines
    ax.axhline(y=bonferroni_threshold, color='red', linestyle='--', 
              linewidth=1.5, label='Bonferroni threshold', 
              alpha=0.7, zorder=1)
    ax.axvline(x=0, color='gray', linestyle='-', linewidth=1, alpha=0.5, zorder=1)
    
    # Label specific points if requested
    if key_labels:
        # Combine both dataframes for labeling
        combined = pd.concat([df1, df2]).drop_duplicates(subset=[match_col])
        
        for label_value in key_labels:
            point_data = combined[combined[match_col] == label_value]
            if not point_data.empty:
                beta = point_data[beta_col].values[0]
                log_p = point_data['log_p'].values[0]
                
                # Smart offset
                offset_x = 10 if beta > 0 else -10
                ha = 'left' if beta > 0 else 'right'
                
                ax.annotate(label_value, (beta, log_p), 
                           xytext=(offset_x, 15), textcoords='offset points',
                           fontsize=11, fontweight='bold', ha=ha,
                           bbox=dict(boxstyle='round,pad=0.4', facecolor='white', 
                                    edgecolor='black', alpha=0.9, linewidth=1.5),
                           zorder=4)
    
    # Formatting
    ax.set_xlabel('Effect Size (β)', fontsize=13, fontweight='bold')
    ax.set_ylabel('-log₁₀(p-value)', fontsize=13, fontweight='bold')
    ax.set_title(title, fontsize=15, fontweight='bold', pad=15)
    ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Set axis limits
    combined_df = pd.concat([df1, df2])
    
    if xlim is not None:
        ax.set_xlim(xlim)
    else:
        x_range = combined_df[beta_col].max() - combined_df[beta_col].min()
        x_margin = x_range * 0.1
        ax.set_xlim(combined_df[beta_col].min() - x_margin, 
                   combined_df[beta_col].max() + x_margin)
    
    if ylim is not None:
        ax.set_ylim(ylim)
    else:
        y_max = max(combined_df['log_p'].max(), bonferroni_threshold * 1.2)
        ax.set_ylim(0, y_max * 1.05)
    
    # Legend
    ax.legend(loc='upper right', frameon=True, fontsize=10, 
             edgecolor='black', fancybox=False)
    
    plt.tight_layout()
    
    # Print summary
    print(f"\nBonferroni threshold: -log10(p) = {bonferroni_threshold:.2f}")
    
    # Save if path provided
    if save_path:
        fig.savefig(save_path, format='svg', dpi=300, bbox_inches='tight')
        print(f"Figure saved to: {save_path}")
    
    plt.show()
    
    return fig, ax


In [ ]:
fig, ax = create_two_dataframe_volcano_plot(
    df1=t1_data,
    df2=latent_data,
    df1_label='T1 Distribution Metrics',
    df2_label='VAE Spatial Features',
    overlap_label='Overlapped',
    match_col='exposure_file',
    show_overlap=True,  # Show 3 categories
    title='MR Results: T1 Distribution and VAE Spatial Features',
    ylim=(0, 15),
    save_path='volcano_comparison_color.svg'
)

In [ ]:
#MR Plot as a volcano plot with labels for top hits
def create_mr_volcano_plot(df):
    # Convert columns to appropriate data types
    df['ivw_estimate'] = df['ivw_estimate'].astype(float)
    df['ivw_pval'] = df['ivw_pval'].astype(float)
    
    # Calculate -log10(p-value)
    df['neg_log10_pval'] = -np.log10(df['ivw_pval'])
    
    # Clean up exposure and outcome names
    df['gene'] = df['exposure_file'].str.replace('_cis_subset.tsv', '')
    
    # Clean outcome names
    df['outcome'] = df['outcome_file'].str.replace('_gwas.txt', '')
    df['outcome'] = df['outcome'].str.replace('imputed_T1_10itr_erroded_gwas_output.', '')
    
    # Create display label
    df['label'] = df['gene'] + ' → ' + df['outcome']
    
    # Categorize p-values for color coding
    def get_pval_category(p):
        if p < 1e-10:
            return "p < 1e-10"
        elif p < 1e-8:
            return "p < 1e-8"
        elif p < 1e-6:
            return "p < 1e-6"
        else:
            return "p ≥ 1e-6"
    
    df['pval_category'] = df['ivw_pval'].apply(get_pval_category)
    
    # Create the volcano plot
    plt.figure(figsize=(12, 8))
    
    # Scatter plot
    scatter = plt.scatter(
        df['ivw_estimate'],
        df['neg_log10_pval'],
        c=df['pval_category'].map({
            "p < 1e-10": "#0047AB",  # Strong blue
            "p < 1e-8": "#4169E1",   # Royal blue
            "p < 1e-6": "#87CEEB",   # Sky blue
            "p ≥ 1e-6": "#D3D3D3"    # Light gray
        }),
        alpha=0.7,
        s=70,  # Point size
        edgecolor='black'
    )

    # Annotate top hits (e.g., top 10 by p-value)
    top_hits = df.nsmallest(20, 'ivw_pval')
    for _, row in top_hits.iterrows():
        plt.annotate(
            row['gene'],
            (row['ivw_estimate'], row['neg_log10_pval']),
            xytext=(-30, 10),
            textcoords='offset points',
            fontsize=9,
            arrowprops=dict(arrowstyle='->', lw=0.5)
        )
        
    
    # Add labels and title
    plt.xlabel('Effect Size (IVW Estimate)', fontsize=14)
    plt.ylabel('-log10(P-value)', fontsize=14)
    plt.title('Volcano Plot of Mendelian Randomization Results', fontsize=16)
    plt.grid(True, alpha=0.3)
    plt.axhline(y=-np.log10(0.05), color='red', linestyle='--', alpha=0.7)  # Significance threshold line
    plt.axvline(x=0, color='gray', linestyle='--', alpha=0.7)  # Null effect line
    plt.tight_layout()
    plt.show()

In [ ]:
create_mr_volcano_plot(t1_data)

In [ ]:
create_mr_volcano_plot(latent_data)

In [ ]:
# Venn diagram of all hits

from matplotlib_venn import venn2

plt.figure(figsize=(8, 6))
venn2(
    subsets = (len(set(latent_data["gene"])) - len(common_genes), 
               len(set(t1_data["gene"])) - len(common_genes), 
               len(common_genes)),
    set_labels = ('Latent MR', 'T1 MR'),
    set_colors=('skyblue', 'lightgreen'),
    alpha=0.7
)
plt.title("Venn Diagram of Significant Genes in MR Analyses", fontsize=16)
plt.tight_layout()
plt.show()

print("Common genes in both MR analyses:", common_genes)

## Nauffal et. al Manual MR

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Your data
data = {
    'SOD2': {'beta': -0.257652053, 'pval': 1.22e-22},
    'ECM1': {'beta': -0.036404599, 'pval': 9.06e-18},
    'SCGN': {'beta': -0.347509426, 'pval': 1.47e-18},
    'RELT': {'beta': 0.076236190, 'pval': 1.75e-14},
    'AGER': {'beta': -0.104012316, 'pval': 1.53e-13},
    'GSTP1': {'beta': 0.090332020, 'pval': 2.16e-12},
    'SUGP1': {'beta': 0.097531462, 'pval': 1.45e-12},
    'LRRC37A2': {'beta': -0.022005599, 'pval': 1.06e-12},
}

def create_mr_forest_plot_from_dict(data_dict, outcome_name="Mean Septal T1", outfile="mr_forest_plot.svg"):
    # Convert dictionary to dataframe
    df = pd.DataFrame.from_dict(data_dict, orient='index').reset_index()
    df.columns = ['gene', 'beta', 'pval']
    
    # Calculate standard error from p-value and beta
    # Formula: SE = |beta| / |Z-score|
    df['z_score'] = stats.norm.ppf(1 - df['pval']/2)  # Two-tailed
    df['se'] = np.abs(df['beta']) / df['z_score']
    
    # Calculate confidence intervals
    df['ci_lower'] = df['beta'] - 1.96 * df['se']
    df['ci_upper'] = df['beta'] + 1.96 * df['se']
    
    # Create display label
    df['label'] = df['gene'] + ' → ' + outcome_name
    
    # Categorize p-values for color coding
    def get_pval_category(p):
        if p < 1e-10:
            return "p < 1e-10"
        elif p < 1e-8:
            return "p < 1e-8"
        elif p < 1e-6:
            return "p < 1e-6"
        else:
            return "p ≥ 1e-6"
    
    df['pval_category'] = df['pval'].apply(get_pval_category)
    
    # Define marker mapping based on p-values
    pval_marker = {
        "p < 1e-10": 'o',
        "p < 1e-8": 'o',
        "p < 1e-6": 'o',
        "p ≥ 1e-6": 's'
    }
    
    df['marker'] = df['pval_category'].map(pval_marker)
    
    # Create a color mapping for p-value categories
    pval_colors = {
        "p < 1e-10": "#0047AB",  # Strong blue
        "p < 1e-8": "#4169E1",   # Royal blue
        "p < 1e-6": "#87CEEB",   # Sky blue
        "p ≥ 1e-6": "#D3D3D3"    # Light gray
    }
    
    # Sort by beta value (or you can sort by pval)
    df_plot = df.sort_values('beta', ascending=False).reset_index(drop=True)
    
    # Create the forest plot
    plt.figure(figsize=(12, 9))
    
    # Plot settings
    spacing = 1
    positions = np.arange(len(df_plot)) * spacing
    
    # Create the plot
    for i, row in df_plot.iterrows():
        # Plot confidence interval line
        plt.plot([row['ci_lower'], row['ci_upper']], [positions[i], positions[i]], 
                 color='black', alpha=0.6, zorder=1, linewidth=1.5)
        
        # Marker size
        marker_size = 600
        
        # Plot the point estimate
        plt.scatter(row['beta'], positions[i], 
                    color=pval_colors[row['pval_category']], 
                    s=marker_size,
                    marker=row['marker'],
                    zorder=2, 
                    edgecolor='black', 
                    linewidth=3,
                    alpha=0.8)
    
    # Add vertical line at x=0 (null effect)
    plt.axvline(x=0, color='gray', linestyle='--', alpha=0.7, zorder=0, linewidth=2)
    
    # Customize the plot
    plt.grid(axis='x', linestyle='--', alpha=0.3, linewidth=1.0)
    plt.xlabel('Effect Size (Beta)', fontsize=14, fontweight='bold')
    plt.title('Mendelian Randomization: Protein → Mean Septal T1', fontsize=16, fontweight='bold')
    
    # Set y-ticks to show gene-outcome pairs
    plt.yticks(positions, df_plot['label'], fontsize=10)
    
    # Create legend for p-value categories
    legend_elements = []
    for pval, color in pval_colors.items():
        marker_shape = pval_marker[pval]
        legend_elements.append(plt.scatter([], [], 
                               s=100, 
                               color=color, 
                               marker=marker_shape,
                               edgecolor='black', 
                               linewidth=1.5,
                               label=pval))
    
    # Add legend
    # plt.legend(
    #     handles=legend_elements, 
    #     labels=list(pval_colors.keys()),
    #     title="P-value",
    #     loc='lower right',
    #     fontsize=10,
    #     title_fontsize=12
    # )
    
    # Set x-axis limits
    x_buffer = 0.05
    x_min = df_plot['ci_lower'].min() - x_buffer
    x_max = df_plot['ci_upper'].max() + x_buffer
    plt.xlim(x_min, x_max)
    
    # Set y-axis limits
    plt.ylim(-1, len(df_plot))
    
    # Tight layout with padding
    plt.tight_layout(pad=2.0)
    
    # Save high-resolution plot
    plt.savefig(outfile, format='svg', dpi=300, bbox_inches='tight')
    print(f"Forest plot saved as {outfile}")
    
    plt.show()
    
    return df

# Run the function
df_result = create_mr_forest_plot_from_dict(data, outfile="septal_t1_mr_forest_plot.svg")

# Display the dataframe to see calculated values
print(df_result)

# PWAS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors

# Define the data
data = """ PA_Phenotype	Protein	Beta	P-Value	FDR	Significant	-log10(P-Value)
Mean_T1	olink_instance_0.cdhr2	-8.674775	1.535543e-32	5.616249e-30	True	31.813738
Mean_T1	olink_instance_0.igfbp2	12.925700	1.700595e-52	2.487970e-49	True	51.769399
Mean_T1	olink_instance_0.lep	-8.834353	9.435460e-51	6.902039e-48	True	50.025237
Mean_T1	olink_instance_0.plat	-12.978140	1.067175e-32	5.204256e-30	True	31.971764
Mean_T1	olink_instance_0.ssc4d	-5.510259	2.387953e-31	6.987151e-29	True	30.621974
T1_Standard_Deviation	olink_instance_0.amigo2	-7.530700	3.315327e-07	4.850323e-04	True	6.479474
T1_Standard_Deviation	olink_instance_0.itih3	4.091225	2.158535e-06	1.223882e-03	True	5.665841
T1_Standard_Deviation	olink_instance_0.ren	2.488676	2.509669e-06	1.223882e-03	True	5.600383
T1_0.25th_Percentile	olink_instance_0.fabp4	-42.562612	9.762074e-38	7.140957e-35	True	37.010458
T1_0.25th_Percentile	olink_instance_0.il1rn	-39.997805	2.197893e-29	9.648554e-27	True	28.657993
T1_0.25th_Percentile	olink_instance_0.inhbc	-49.558725	2.638019e-29	9.648554e-27	True	28.578722
T1_0.25th_Percentile	olink_instance_0.lep	-31.526774	1.072046e-55	1.568403e-52	True	54.969787
T1_0.25th_Percentile	olink_instance_0.plat	-39.660234	1.156077e-26	3.382681e-24	True	25.937013
T1_1th_Percentile	olink_instance_0.fabp4	-42.469799	9.665034e-49	7.069972e-46	True	48.014797
T1_1th_Percentile	olink_instance_0.il1rn	-37.391884	2.461603e-33	9.003314e-31	True	32.608782
T1_1th_Percentile	olink_instance_0.inhbc	-47.589975	4.930705e-35	2.404541e-32	True	34.307091
T1_1th_Percentile	olink_instance_0.lep	-31.735292	1.127067e-73	1.648899e-70	True	72.948050
T1_1th_Percentile	olink_instance_0.plat	-37.556376	6.035539e-31	1.765999e-28	True	30.219284
T1_25th_Percentile	olink_instance_0.cdhr2	-6.589703	6.090157e-24	2.227475e-21	True	23.215371
T1_25th_Percentile	olink_instance_0.fabp4	-9.084864	2.173646e-25	1.060015e-22	True	24.662811
T1_25th_Percentile	olink_instance_0.igfbp2	9.397035	4.178291e-35	3.056420e-32	True	34.379001
T1_25th_Percentile	olink_instance_0.lep	-6.822715	4.352812e-38	6.368164e-35	True	37.361230
T1_25th_Percentile	olink_instance_0.plat	-9.496023	2.281707e-22	6.676275e-20	True	21.641740
T1_50th_Percentile	olink_instance_0.cdhr2	-5.596714	5.325706e-19	1.947877e-16	True	18.273623
T1_50th_Percentile	olink_instance_0.cntn3	-13.021221	6.189324e-18	1.810996e-15	True	17.208357
T1_50th_Percentile	olink_instance_0.igfbp2	8.954105	1.540836e-34	2.254243e-31	True	33.812244
T1_50th_Percentile	olink_instance_0.lep	-5.012503	8.227506e-23	6.018421e-20	True	22.084732
T1_50th_Percentile	olink_instance_0.plat	-8.463526	1.887930e-19	9.206805e-17	True	18.724014
T1_75th_Percentile	olink_instance_0.cntn3	-18.747925	9.757956e-23	7.137944e-20	True	22.010641
T1_75th_Percentile	olink_instance_0.igfbp2	12.071873	5.534695e-39	8.097260e-36	True	38.256906
T1_75th_Percentile	olink_instance_0.lep	-6.165401	1.270434e-21	4.646613e-19	True	20.896048
T1_75th_Percentile	olink_instance_0.plat	-11.256514	2.624832e-21	7.680258e-19	True	20.580898
T1_75th_Percentile	olink_instance_0.ssc4d	-4.963407	6.193127e-22	3.020181e-19	True	21.208090
T1_99th_Percentile	olink_instance_0.cdhr2	-20.038075	3.985247e-30	1.166083e-27	True	29.399545
T1_99th_Percentile	olink_instance_0.cpm	-43.276571	8.022580e-31	2.934259e-28	True	30.095686
T1_99th_Percentile	olink_instance_0.igfbp2	29.013986	3.130607e-46	2.290039e-43	True	45.504372
T1_99th_Percentile	olink_instance_0.lep	-20.640979	2.906432e-48	4.252110e-45	True	47.536640
T1_99th_Percentile	olink_instance_0.ssc4d	-13.173085	3.167516e-31	1.544692e-28	True	30.499281
T1_99.75th_Percentile	olink_instance_0.cdhr2	-28.918453	1.576494e-24	4.612821e-22	True	23.802308
T1_99.75th_Percentile	olink_instance_0.cpm	-63.221679	1.514309e-25	5.538586e-23	True	24.819785
T1_99.75th_Percentile	olink_instance_0.fabp4	-40.355362	1.064819e-26	5.192769e-24	True	25.972724
T1_99.75th_Percentile	olink_instance_0.igfbp2	41.386566	2.500281e-36	1.828956e-33	True	35.602011
T1_99.75th_Percentile	olink_instance_0.lep	-33.737127	1.344854e-49	1.967521e-46	True	48.871325"""

# Parse the data into a DataFrame
df = pd.DataFrame([x.split('\t') for x in data.strip().split('\n')])
header = df.iloc[0]
df = df[1:]  # Remove header row
df.columns = header

# Convert columns to appropriate types
df['Beta'] = df['Beta'].astype(float)
df['P-Value'] = df['P-Value'].astype(float)
df['-log10(P-Value)'] = df['-log10(P-Value)'].astype(float)

# Clean up protein names
df['Protein'] = df['Protein'].str.replace('olink_instance_0.', '')

# Order phenotypes in a logical sequence
phenotype_order = [
    'Mean_T1', 
    'T1_Standard_Deviation', 
    'T1_0.25th_Percentile', 
    'T1_1th_Percentile', 
    'T1_25th_Percentile', 
    'T1_50th_Percentile', 
    'T1_75th_Percentile', 
    'T1_99th_Percentile',
    'T1_99.75th_Percentile'
]

# Create a pivot table for the -log10(P-Value)
significance_pivot = df.pivot_table(
    values='-log10(P-Value)', 
    index='Protein', 
    columns='PA_Phenotype'
)

# Reorder the columns according to our logical sequence
significance_pivot = significance_pivot[phenotype_order]

# Create a pivot table for Beta values (will be used for annotations)
beta_pivot = df.pivot_table(
    values='Beta', 
    index='Protein', 
    columns='PA_Phenotype'
)
beta_pivot = beta_pivot[phenotype_order]

# Set up the matplotlib figure
plt.figure(figsize=(14, 10))

# Create a custom colormap that goes from white to green
cmap = mcolors.LinearSegmentedColormap.from_list("GreenGradient", ["#ffffff", "#e0f3db", "#a8ddb5", "#43a2ca", "#0868ac"])
# cmap of red to blue
cmap = mcolors.LinearSegmentedColormap.from_list("RedBlueGradient", ["#e0f3db", "#f7fbff", "#deebf7", "#9ecae1", "#3182bd"])
# cmap = sns.color_palette("RdBu", as_cmap=True)


# Plot the heatmap
ax = sns.heatmap(
    significance_pivot,
    cmap=cmap,
    annot=True,              # Show values in cells
    fmt=".1f",               # Format for the annotations (1 decimal place)
    linewidths=.5,           # Width of the lines that divide each cell
    cbar_kws={'label': '-log10(P-Value)'},
    vmin=5,                  # Minimum value for the colormap
    vmax=75,                 # Maximum value for the colormap
    square=True,             # Make cells square-shaped
    linecolor='white',
    annot_kws={"size": 8}    # Font size of the annotation text
)

# Improve the appearance
plt.title('Protein Phenotype Association Significance Heatmap', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)

# Format the x-axis labels to be more readable
labels = [item.get_text().replace('T1_', '') for item in ax.get_xticklabels()]
ax.set_xticklabels(labels)

# Add a note about interpretation
plt.figtext(0.5, 0.01, 
           "Note: Higher values (darker colors) indicate greater statistical significance. Beta values (β) show effect direction and magnitude.", 
           ha="center", fontsize=10, style='italic')

# Tight layout and save
plt.tight_layout()
plt.savefig('protein_significance_heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig('protein_significance_heatmap.pdf', format='pdf', bbox_inches='tight')

print("Heatmap created and saved as 'protein_significance_heatmap.png' and 'protein_significance_heatmap.pdf'")

# Show the plot
plt.show()

# Updated, more comprehensive MR Results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
old_data = pd.read_csv("./data/MR_significant_hits.txt", sep='\t')
new_data = pd.read_csv("./data/MR_2_significant_hits.txt", sep='\t')

In [ ]:
#check overlap in exposure files in old and new data

old_exposures = set(old_data['exposure_file'].str.replace('_cis_subset.tsv', ''))
new_exposures = set(new_data['exposure_file'].str.replace('_cis_subset.tsv', ''))
overlap_exposures = old_exposures.intersection(new_exposures)
print(f"Number of overlapping exposures: {len(overlap_exposures)}")

#non overlapping exposures
non_overlap_exposures = old_exposures.difference(new_exposures)
print(f"Non-overlapping exposures: {non_overlap_exposures}")

In [ ]:
# analyse the new data

new_data['exposure_file'] = new_data['exposure_file'].str.replace('_cis_subset.tsv', '')
new_data['outcome_file'] = new_data['outcome_file'].str.replace('_gwas.txt', '')

In [ ]:
new_data[new_data["exposure_file"] == "PGF"]

In [ ]:
threshold = 0.05/(3000*25)
print(f"Significance threshold for multiple testing correction: {threshold:.4e}")

print(0.000006	< threshold)

In [ ]:
highly_significant_data = new_data[new_data['ivw_pval'] < 0.05/(2000*25)]

In [ ]:
overlap_exposures_high = old_exposures.intersection(highly_significant_data['exposure_file'].str.replace('_cis_subset.tsv', ''))
print(f"Number of overlapping exposures with highly significant data: {len(overlap_exposures_high)}")
non_overlap_exposures_high = old_exposures.difference(highly_significant_data['exposure_file'].str.replace('_cis_subset.tsv', ''))
print(f"Non-overlapping exposures: {non_overlap_exposures_high}")


In [ ]:
#sort highly_significant_data dataframe by exposure file - to clump the phenotypes associated with the same exposure
highly_significant_data_sorted = highly_significant_data.sort_values(by=['ivw_pval', 'exposure_file'], ascending=[True, True])
highly_significant_data_sorted

In [ ]:
highly_significant_data_sorted['exposure_file'].unique()

In [ ]:
# seperate which causal exposures are assocated with T1 phenotypes and which are associated with latent phenotypes

latent_exposures = highly_significant_data_sorted[highly_significant_data_sorted['outcome_file'].str.contains('Latent')]
T1_exposures = highly_significant_data_sorted[highly_significant_data_sorted['outcome_file'].str.contains('T1')]

In [ ]:
latent_hits = set(latent_exposures['exposure_file'].unique())
T1_hits = set(T1_exposures['exposure_file'].unique())

print("Latent hits: " + str(latent_hits))
print("T1 hits: " + str(T1_hits))

In [ ]:
overlap_exposures_T1_latent = latent_hits.intersection(T1_hits)
print(f"Number of overlapping exposures between T1 and latent: {len(overlap_exposures_T1_latent)}")
print("Overlapping exposures between T1 and latent: " + str(overlap_exposures_T1_latent))

In [ ]:
# get direction of effect for each exposure

def get_direction_of_effect(exposure, outcome, data):
    # Filter the data for the specific exposure and outcome
    filtered_data = data[(data['exposure_file'].str.replace('_cis_subset.tsv', '') == exposure) & 
                         (data['outcome_file'].str.replace('_gwas.txt', '') == outcome)]
    
    # Check if there are any results
    if not filtered_data.empty:
        # Get the beta value
        beta_value = filtered_data['ivw_estimate'].values[0]
        return beta_value
    else:
        return None

# Get the direction of effect for each exposure
for hit in T1_hits:
    for outcome in T1_exposures['outcome_file'].unique():
        direction = get_direction_of_effect(hit, outcome, highly_significant_data_sorted)
        if direction is not None:
            print(f"Exposure: {hit}, Outcome: {outcome}, Direction of Effect: {direction > 0 and 'Positive' or 'Negative'}, Beta: {direction:.4f}")

In [ ]:
for hit in latent_hits:
    for outcome in latent_exposures['outcome_file'].unique():
        direction = get_direction_of_effect(hit, outcome, highly_significant_data_sorted)
        if direction is not None:
            print(f"Exposure: {hit}, Outcome: {outcome}, Beta: {direction:.4f}")

In [ ]:
# print out a table for the exposures which just lits the exposure and whether the effect is positive or negative
T1_exposure_effects = []
for hit in T1_hits:
    for outcome in T1_exposures['outcome_file'].unique():
        direction = get_direction_of_effect(hit, outcome, highly_significant_data_sorted)
        if direction is not None:
            if direction > 0:
                direction = "Pro-fibrotic"
            elif direction < 0:
                direction = "Anti-fibrotic"
            else:
                direction = "No Effect"
            T1_exposure_effects.append((hit, direction))
# Create a DataFrame from the list
T1_exposure_effects_df = pd.DataFrame(T1_exposure_effects, columns=['Exposure', 'Direction of Effect'])

In [ ]:
T1_exposure_effects_df.drop_duplicates(subset=['Exposure'], inplace=True)
T1_exposure_effects_df

# COLOC Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
latent_coloc = pd.read_csv("./data/coloc_results_latent_task01_of01.tsv.gz")
t1_coloc = pd.read_csv("./data/coloc_results_task01_of01.tsv.gz")

all_coloc = pd.concat([latent_coloc, t1_coloc], ignore_index=True)

In [ ]:
all_coloc.sort_values(by=['pp_h4'], ascending=False, inplace=True)
all_coloc.head(n=20)

In [ ]:
def get_strong_colocalizations(df, threshold=0.7):
    """Extract strong colocalization results"""
    strong = df[df['pp_h4'] > threshold].sort_values('pp_h4', ascending=False)
    return strong[['exposure', 'outcome', 'pp_h4', 'pp_h3', 'n_snps']]

def get_separate_causal(df, threshold=0.7):
    """Extract separate causal variant results"""
    separate = df[df['pp_h3'] > threshold].sort_values('pp_h3', ascending=False)
    return separate[['exposure', 'outcome', 'pp_h4', 'pp_h3', 'n_snps']]

def gene_summary(df):
    """Summarize results by gene"""
    gene_stats = df.groupby('exposure').agg({
        'pp_h4': ['count', 'max', 'mean'],
        'pp_h3': 'max',
        'n_snps': 'first'
    }).round(3)
    
    gene_stats.columns = ['n_tests', 'max_pp_h4', 'mean_pp_h4', 'max_pp_h3', 'n_snps']
    gene_stats['strong_coloc'] = df.groupby('exposure')['pp_h4'].apply(lambda x: sum(x > 0.8))
    
    return gene_stats.sort_values('max_pp_h4', ascending=False)

strong_coloc = get_strong_colocalizations(all_coloc)
separate_causal = get_separate_causal(all_coloc)
gene_summary_df = gene_summary(all_coloc)
# Display the results
print("Strong Colocalizations (pp_h4 > 0.7):")
print(strong_coloc.head(20))
print("\nSeparate Causal Variants (pp_h3 > 0.7):")
print(separate_causal.head(20))
print("\nGene Summary Statistics:")
print(gene_summary_df.head(20))

# SMR-HEIDI Analysis

In [ ]:
summary_hits = pd.read_csv("./data/MR_experiment/summary_hits.txt", sep='\t')

#remove all rows where the Traist contrain 'run3_imputation_ukb24983'
summary_hits = summary_hits[~summary_hits['Trait'].str.contains('run3_imputation_ukb24983')]

In [ ]:
from statsmodels.stats.multitest import multipletests

# FDR correction (less conservative)
_, p_fdr, _, _ = multipletests(summary_hits['p_SMR'], method='fdr_bh')
summary_hits['p_SMR_fdr'] = p_fdr

# Compare thresholds
print(f"Bonferroni (0.05/{len(summary_hits)}): {(summary_hits['p_SMR'] < 0.05/len(summary_hits)).sum()} hits")
print(f"FDR < 0.05: {(p_fdr < 0.05).sum()} hits") 
print(f"Nominal p < 0.01: {(summary_hits['p_SMR'] < 0.01).sum()} hits")

# Use FDR correction for nominal hits
causal = summary_hits[(summary_hits['p_SMR'] < 0.01) & 
                     (summary_hits['p_HEIDI'] > 0.05) & 
                     (summary_hits['nsnp_HEIDI'] >= 3)]

In [ ]:
# Top causal gene-trait pairs by tissue
results = causal.groupby(['Trait', 'Gene', 'Tissue']).agg({
    'p_SMR': 'min',
    'b_SMR': 'first',
    'topSNP': 'first'
}).reset_index().sort_values('p_SMR')

# Summary by trait
trait_summary = causal.groupby('Trait').agg({
    'Gene': 'nunique',
    'Tissue': lambda x: list(x.unique())
}).rename(columns={'Gene': 'n_genes'})

In [ ]:
gene_trait_ranking = causal.groupby(['Trait', 'Gene']).agg({
    'p_SMR': 'min',
    'b_SMR': lambda x: x.iloc[x.abs().argmax()],  # Effect with largest magnitude
    'Tissue': lambda x: list(x),
    'topSNP': 'first'
}).reset_index().sort_values('p_SMR')

print(f"\nTop 20 Gene-Trait Causal Relationships:")
print(gene_trait_ranking.head(20)[['Trait', 'Gene', 'p_SMR', 'b_SMR']])

In [ ]:
tissue_effects = causal.pivot_table(
    index=['Trait', 'Gene'], 
    columns='Tissue', 
    values='b_SMR', 
    fill_value=0
).abs()

In [ ]:
tissue_summary = causal.groupby('Tissue').agg({
    'Gene': 'nunique',
    'Trait': 'nunique', 
    'b_SMR': lambda x: abs(x).mean()
}).rename(columns={'Gene': 'n_genes', 'Trait': 'n_traits', 'b_SMR': 'mean_effect'}).sort_values('n_genes', ascending=False)

print(f"\nMost Important Tissues:")
print(tissue_summary.head(10))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style
plt.style.use('default')
sns.set_style("whitegrid")

# Create the figure
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# COLOC data (strong colocalizations pp_h4 > 0.7)
coloc_genes = ['LRRC37A2', 'PDE5A', 'CEACAM21', 'PGF', 'CTSS']
coloc_pp_h4 = [1.0, 0.934, 0.910, 0.877, 0.800]
coloc_traits = ['T1_75th', 'Latent_9', 'Latent_1', 'T1_99th', 'T1_99th']

# Create scatter plot with very large points
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
scatter = ax.scatter(range(len(coloc_genes)), coloc_pp_h4, 
                     s=500,  # Very large points
                     c=colors, 
                     alpha=0.8, 
                     edgecolors='black', 
                     linewidth=2)

# Customize the plot
ax.set_xticks(range(len(coloc_genes)))
ax.set_xticklabels(coloc_genes, fontsize=12, fontweight='bold')
ax.set_ylabel('Posterior Probability (PP4)', fontsize=14, fontweight='bold')
ax.set_title('Colocalization Analysis: GWAS + eQTL/pQTL', fontsize=16, fontweight='bold', pad=20)

# Add threshold line
ax.axhline(y=0.8, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Strong Evidence (PP4 = 0.8)')

# Set y-axis limits
ax.set_ylim(0.75, 1.02)

# Add trait labels above each point
for i, (gene, pp4, trait) in enumerate(zip(coloc_genes, coloc_pp_h4, coloc_traits)):
    ax.text(i, pp4 + 0.02, trait, ha='center', va='bottom', 
            fontsize=11, fontweight='bold', 
            bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8))

# Customize grid and spines
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

# Add legend
ax.legend(loc='lower right', fontsize=12)

# Adjust layout
plt.tight_layout()
plt.show()

# Print summary
print("\nColocalization Results Summary:")
print("="*40)
for gene, pp4, trait in zip(coloc_genes, coloc_pp_h4, coloc_traits):
    evidence = "Very Strong" if pp4 > 0.95 else "Strong" if pp4 > 0.8 else "Moderate"
    print(f"{gene:10} | {trait:12} | PP4: {pp4:.3f} | {evidence}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches

# Set up the figure with high DPI for quality
plt.figure(figsize=(20, 12), dpi=150)

# Define diseases (rows)
diseases = [
    'Normal',
    'Hypertension',
    'Type 2 Diabetes',
    'Ischemic Heart Disease',
    'Myocardial Infarction',
    'Valvular Disease',
    'Type 1 Diabetes',
    'Non-Ischemic Heart Disease',
    'Sarcoidosis'
]

# Define features (columns)
features = [
    'Mean T1',
    'T1 0-25th %ile',
    'T1 25th-50th %ile',
    'T1 50th-75th %ile',
    'T1 75th %ile',
    'T1 >75th %ile',
    'T1 SD/Mean',
    'T1 Skewness',
    'T1 Standard Dev',
    'Latent 1',
    'Latent 2',
    'Latent 3',
    'Latent 4',
    'Latent 5',
    'Latent 6',
    'Latent 7',
    'Latent 9',
    'Latent 10',
    'Latent 11',
    'Latent 14',
    'Latent 15'
]

# Data matrix (-log10(p) values)
data = np.array([
    [2.63, np.nan, np.nan, 1.83, np.nan, np.nan, np.nan, np.nan, 2.37, 3.87, 7.36, 3.33, 1.58, 5.02, np.nan, np.nan, 4.76, 10.44, 9.01, np.nan, 2.25],
    [8.63, 6.93, 7.21, 6.34, 3.12, 7.67, 4.46, 6.02, np.nan, 2.71, 2.02, np.nan, np.nan, 1.68, np.nan, 4.92, 5.86, 5.86, np.nan, 2.54, 1.73],
    [1.77, np.nan, np.nan, np.nan, 3.84, 3.83, np.nan, np.nan, 2.86, 4.37, np.nan, np.nan, 6.21, np.nan, np.nan, np.nan, 1.69, 6.16, 2.32, 3.76, np.nan],
    [5.29, 2.02, 2.89, 5.66, 2.95, 4.19, np.nan, np.nan, 3.67, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
    [2.66, np.nan, 2.44, 4.37, 1.54, 1.81, np.nan, 1.49, np.nan, 4.00, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
    [np.nan, np.nan, np.nan, np.nan, 1.79, np.nan, 2.07, 2.00, 1.52, np.nan, 1.85, np.nan, np.nan, 2.40, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
    [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 2.10, np.nan, 1.79, 2.15, np.nan, np.nan, np.nan, 2.08, np.nan, 1.61, np.nan],
    [np.nan, np.nan, np.nan, np.nan, 2.37, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 1.93, np.nan, np.nan, np.nan, 3.64, np.nan, np.nan, np.nan],
    [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 1.67, np.nan, np.nan, np.nan, np.nan, np.nan]
])

# Create custom colormap
colors = ['#ffffff', '#ffb6c1', '#ff69b4', '#b30000', '#7f0000']
n_bins = 100
cmap = sns.blend_palette(colors, n_colors=n_bins, as_cmap=True)

# Create the heatmap
ax = sns.heatmap(data, 
                 xticklabels=features,
                 yticklabels=diseases,
                 cmap=cmap,
                 vmin=0,
                 vmax=11,
                 center=5.5,
                 annot=True,
                 fmt='.2f',
                 cbar_kws={'label': '-log₁₀(p-value)', 'shrink': 0.8},
                 linewidths=0.5,
                 linecolor='lightgray',
                 square=False)

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=11)

# Add title and subtitle
plt.suptitle('Cardiac Disease Association Analysis: Confounder-Adjusted T1 Mapping versus Latent Dimension',
             fontsize=16, fontweight='bold', y=0.98)

# Add vertical line to separate T1 features from Latent features
ax.axvline(x=9, color='gray', linestyle='--', linewidth=2, alpha=0.7)

# Add section labels
ax.text(4.5, -1.2, 'T1 Relaxation Time Scalars', 
        ha='center', fontsize=12, fontweight='bold', color='#c7254e')
ax.text(15, -1.2, 'VAE Latent Dimensions', 
        ha='center', fontsize=12, fontweight='bold', color='#6f42c1')

# Create custom legend at the bottom - BIGGER AND HORIZONTAL
legend_elements = [
    mpatches.Patch(facecolor='#ffffff', edgecolor='gray', label='Not Significant (-log₁₀(p) < 1.3)'),
    mpatches.Patch(facecolor='#ffb6c1', label='p < 0.05 (-log₁₀(p): 1.3-2.0)'),
    mpatches.Patch(facecolor='#ff69b4', label='p < 0.01 (-log₁₀(p): 2.0-3.0)'),
    mpatches.Patch(facecolor='#b30000', label='p < 0.001 (-log₁₀(p): 3.0-4.0)'),
    mpatches.Patch(facecolor='#7f0000', label='p < 0.0001 (-log₁₀(p) ≥ 4.0)')
]

ax.legend(handles=legend_elements, 
          loc='upper center', 
          bbox_to_anchor=(0.5, -0.28), 
          title='Significance Level', 
          frameon=True, 
          fontsize=13,
          title_fontsize=15,
          ncol=5,
          columnspacing=2.5,
          handlelength=3,
          handleheight=2.5,
          edgecolor='black',
          fancybox=True,
          shadow=True)

# Adjust layout to prevent label cutoff
plt.tight_layout(rect=[0, 0.08, 1, 0.96])

plt.savefig('cardiac_disease_heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig('cardiac_disease_heatmap.pdf', bbox_inches='tight')
print("Heatmap saved as 'cardiac_disease_heatmap.png' and 'cardiac_disease_heatmap.pdf'")
plt.show()

In [ ]:
#quick check of snp files

snp_old = pd.read_csv("./data/downloads/calls_22282.QC.snplist", sep="\t", header=None)
snp_new = pd.read_csv("./data/downloads/calls_22282_SR.QC.snplist", sep="\t", header=None)

In [ ]:
#quick check of snp files
print(f"Old SNP file total: {len(snp_old)}")
print(f"New SNP file total: {len(snp_new)}")

common_snps = set(snp_old[0]) & set(snp_new[0])
print(f"Number of common SNPs: {len(common_snps)}")

unique_old = set(snp_old[0]) - set(snp_new[0])
unique_new = set(snp_new[0]) - set(snp_old[0])
print(f"Unique to old SNP file: {len(unique_old)}")
print(f"Unique to new SNP file: {len(unique_new)}")

# Cox Hazard Models

In [ ]:
# ============================
#   FIGURE 1 & FIGURE 2 SCRIPT
#   Survival Analysis Summary
# ============================

import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------------------------------
# Load and filter data
# ----------------------------------------------------
df = pd.read_csv("./data/survival_analysis_results.csv")

# Remove hematocrit-regressed phenotypes
df = df[~df["Phenotype"].str.contains("_regressed_hematocrit", case=False, na=False)]
df = df[~df["Phenotype"].str.contains("id", case=False, na=False)]

# -log10(p)
df["neglog10_p"] = -np.log10(df["Logrank_P"])

# clean phenotype labels
df["CleanPhenotype"] = (
    df["Phenotype"]
    .str.replace("_", " ")
    .str.replace("T1map", "T1 map")
)

# ----------------------------------------------------
# Select top N phenotypes for the main figures
# ----------------------------------------------------
TOP_N = 10   # change this number if you want 6, 10, etc.

df_main = df.sort_values("Logrank_P").head(TOP_N)
df_supp = df.sort_values("Logrank_P").iloc[TOP_N:]   # everything else

# ----------------------------------------------------
# FIGURE 1: Top survival associations (-log10 p)
# ----------------------------------------------------
fig1 = plt.figure(figsize=(8, 5))

sorted_main = df_main.sort_values("neglog10_p", ascending=False)

plt.barh(
    sorted_main["CleanPhenotype"],
    sorted_main["neglog10_p"],
    height=0.6
)
plt.xlabel("-log10(p)")
plt.title("Figure 1. Top Survival Associations Across Fibrosis Phenotypes")
plt.gca().invert_yaxis()  # strongest at top
plt.tight_layout()

plt.show()

# ----------------------------------------------------
# FIGURE 2: Forest plot of Hazard Ratios for top phenotypes
# ----------------------------------------------------
fig2 = plt.figure(figsize=(8, 6))

sorted_main = df_main.sort_values("Hazard_Ratio", ascending=True)
y = np.arange(len(sorted_main))

plt.errorbar(
    x=sorted_main["Hazard_Ratio"],
    y=y,
    xerr=[
        sorted_main["Hazard_Ratio"] - sorted_main["HR_95CI_Lower"],
        sorted_main["HR_95CI_Upper"] - sorted_main["Hazard_Ratio"]
    ],
    fmt='o',
    capsize=4
)

plt.yticks(y, sorted_main["CleanPhenotype"])
plt.axvline(1, linestyle="--", linewidth=1)
plt.xlabel("Hazard Ratio")
plt.title("Figure 2. Hazard Ratios for Top Fibrosis Phenotypes")

plt.tight_layout()
plt.show()

# ----------------------------------------------------
# SUPPLEMENTARY FIGURES
#   - full -log10(p) panel
#   - full HR forest plot
# ----------------------------------------------------
figS1 = plt.figure(figsize=(7, max(4, len(df_supp)*0.3)))

sorted_supp = df_supp.sort_values("neglog10_p", ascending=False)

plt.barh(
    sorted_supp["CleanPhenotype"],
    sorted_supp["neglog10_p"],
    height=0.6
)
plt.xlabel("-log10(p)")
plt.title("Supplementary Figure 1. Full Survival Associations")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

figS2 = plt.figure(figsize=(7, max(4, len(df_supp)*0.3)))

sorted_supp = df_supp.sort_values("Hazard_Ratio", ascending=True)
y = np.arange(len(sorted_supp))

plt.errorbar(
    x=sorted_supp["Hazard_Ratio"],
    y=y,
    xerr=[
        sorted_supp["Hazard_Ratio"] - sorted_supp["HR_95CI_Lower"],
        sorted_supp["HR_95CI_Upper"] - sorted_supp["Hazard_Ratio"]
    ],
    fmt='o',
    capsize=4
)
plt.axvline(1, linestyle="--", linewidth=1)
plt.yticks(y, sorted_supp["CleanPhenotype"])
plt.xlabel("Hazard Ratio")
plt.title("Supplementary Figure 2. Full Forest Plot")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# -------------------------
# 1. Create dataframe manually from your provided data
# -------------------------

data = {
    "Feature": [
        "Latent_11", "Latent_5", "T1_75th_Percentile", 
        "Latent_1", "Latent_7", "Latent_10",
        "Mean_T1_regressed_hct_htn", "T1_25th_Percentile"
    ],
    "HR": [0.536282, 1.711409, 1.595289, 0.626891, 0.401304,
           1.567740, 1.567515, 0.662],
    "Lower_CI": [0.415759, 1.323101, 1.230102, 0.483385,
                 0.238132, 1.207996, 1.207823, 0.51],
    "Upper_CI": [0.691745, 2.213677, 2.068892, 0.812999,
                 0.676283, 2.034615, 2.034323, 0.86],
    "Type": ["VAE","VAE","T1","VAE","VAE","VAE","T1","T1"]
}

df = pd.DataFrame(data)

# Sort for clean plotting
df = df.sort_values("HR")

# -------------------------
# 2. Forest plot
# -------------------------

fig, ax = plt.subplots(figsize=(6, 5))

y_positions = range(len(df))
colors = df["Type"].map({"VAE": "tab:blue", "T1": "tab:orange"})

ax.errorbar(
    df["HR"], y_positions,
    xerr=[df["HR"] - df["Lower_CI"], df["Upper_CI"] - df["HR"]],
    fmt='o', color='black', ecolor=colors, elinewidth=3, capsize=5
)

# Reference HR = 1 line
ax.axvline(1, color='gray', linestyle='--')

ax.set_yticks(y_positions)
ax.set_yticklabels(df["Feature"])
ax.set_xlabel("Hazard Ratio (log scale)")
ax.set_xscale('log')
ax.set_title("Top Predictors of Mortality: VAE vs T1")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

vae_data = {
    "Dimension": [
        "Latent_5","Latent_11","Latent_15","Latent_7","Latent_10",
        "Latent_1","Latent_3","Latent_6","Latent_0","Latent_2",
        "Latent_4","Latent_8","Latent_12","Latent_13","Latent_14","Latent_9"
    ],
    "C_index_improvement": [
        0.036384, 0.024273, 0.020577, 0.018707, 0.016671,
        0.015479, 0.013138, 0.011665, 0.008510, 0.007820,
        0.007648, 0.001147, 0.000078, -0.000139, -0.000447, -0.000682
    ],
    "LR_p": [
        0.000082, 0.001674, 0.004048, 0.004529, 0.001518,
        0.003124, 0.018459, 0.072892, 0.081671, 0.139624,
        0.074022, 0.428041, 0.715665, 0.949766, 0.692020, 0.778474
    ]
}

df2 = pd.DataFrame(vae_data)

# Significance coloring
df2["Significant"] = df2["LR_p"] < 0.05

df2 = df2.sort_values("C_index_improvement", ascending=False)

# -------------------------
# Bar plot
# -------------------------

fig, ax = plt.subplots(figsize=(7, 6))

colors = df2["Significant"].map({True: "tab:blue", False: "lightgray"})

ax.barh(df2["Dimension"], df2["C_index_improvement"], color=colors)

ax.set_xlabel("C-index Improvement Over Mean T1")
ax.set_title("Predictive Improvement from Individual VAE Latent Dimensions")

# Vertical reference line
ax.axvline(0, color="black", linewidth=1)

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(5,3))

ax.bar(["Mean T1", "Mean T1 + All 16 Latents"], [0.547, 0.614],
       color=["gray", "tab:blue"])

ax.set_ylabel("C-index")
ax.set_title("Nested Model Comparison")

plt.tight_layout()
plt.show()
